In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
df = pd.read_csv(Path.cwd().parent / "data" / "spy_eod_202303.txt")
df.columns = [k.strip(' ') for k in df.columns]
df

,[QUOTE_UNIXTIME],[QUOTE_READTIME],[QUOTE_DATE],[QUOTE_TIME_HOURS],[UNDERLYING_LAST],[EXPIRE_DATE],[EXPIRE_UNIX],[DTE],[C_DELTA],[C_GAMMA],...,[P_LAST],[P_DELTA],[P_GAMMA],[P_VEGA],[P_THETA],[P_RHO],[P_IV],[P_VOLUME],[STRIKE_DISTANCE],[STRIKE_DISTANCE_PCT]
0,1677704400,2023-03-01 16:00,2023-03-01,16.0,394.76,2023-03-01,1677704400,0.00,1.00000,0.00000,...,0.01,-0.00069,0.00008,-0.00004,-0.00482,0.00000,1.486360,0.000000,74.8,0.189
1,1677704400,2023-03-01 16:00,2023-03-01,16.0,394.76,2023-03-01,1677704400,0.00,1.00000,0.00000,...,0.02,-0.00080,0.00011,0.00034,-0.00529,0.00000,1.282980,0.000000,64.8,0.164
2,1677704400,2023-03-01 16:00,2023-03-01,16.0,394.76,2023-03-01,1677704400,0.00,1.00000,0.00000,...,0.00,-0.00075,0.00014,0.00100,-0.00509,-0.00046,1.162910,,58.8,0.149
3,1677704400,2023-03-01 16:00,2023-03-01,16.0,394.76,2023-03-01,1677704400,0.00,1.00000,0.00000,...,0.00,-0.00128,0.00020,0.00096,-0.00470,0.00000,1.143500,,57.8,0.146
4,1677704400,2023-03-01 16:00,2023-03-01,16.0,394.76,2023-03-01,1677704400,0.00,1.00000,0.00000,...,0.00,-0.00050,0.00015,0.00027,-0.00523,-0.00017,1.123140,,56.8,0.144
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91230,1680292800,2023-03-31 16:00,2023-03-31,16.0,409.39,2025-12-19,1766178000,994.04,0.08235,0.00152,...,0.00,-1.00000,0.00000,0.00000,0.00000,0.00000,,,220.6,0.539
91231,1680292800,2023-03-31 16:00,2023-03-31,16.0,409.39,2025-12-19,1766178000,994.04,0.08401,0.00146,...,0.00,-1.00000,0.00000,0.00000,0.00000,0.00000,,,225.6,0.551
91232,1680292800,2023-03-31 16:00,2023-03-31,16.0,409.39,2025-12-19,1766178000,994.04,0.06465,0.00132,...,0.00,-1.00000,0.00000,0.00000,0.00000,0.00000,,,230.6,0.563
91233,1680292800,2023-03-31 16:00,2023-03-31,16.0,409.39,2025-12-19,1766178000,994.04,0.07999,0.00138,...,0.00,-1.00000,0.00000,0.00000,0.00000,0.00000,,,235.6,0.576


In [18]:
quotes = df["[QUOTE_READTIME]"]
print(quotes.sort_values())

0         2023-03-01 16:00
2538      2023-03-01 16:00
2539      2023-03-01 16:00
2540      2023-03-01 16:00
2541      2023-03-01 16:00
               ...        
88717     2023-03-31 16:00
88718     2023-03-31 16:00
88719     2023-03-31 16:00
88707     2023-03-31 16:00
91234     2023-03-31 16:00
Name: [QUOTE_READTIME], Length: 91235, dtype: object


In [ ]:
def second_difference(date1, date2):
    items_first = date1.split("-")
    items_second = date2.split("-")
    year1 = int(items_first[0])
    month1 = int(items_first[1])
    day1 = int(items_first[2])
    year2 = int(items_second[0])
    month2 = int(items_second[1])
    day2 = int(items_second[2])
    return (
        3600 * 24 * 365 * abs(year1 - year2) +
        3600 * 24 * 30 * abs(month1 - month2) +
        3600 * 24 * (day1 - day2)
    )

start_day = '2023-03-26'
selected_strikes = [120.0, 130.0, 140.0, 150.0]
selected_expiries = [' 2023-06-16', ' 2023-03-31', ' 2023-04-21', ' 2023-05-19']
selected_expiries = df[df["[EXPIRE_DATE]"].isin(selected_expiries)]["[EXPIRE_DATE]"].unique()

from itertools import product
common_idx = set()
for strike, expiry in product(selected_strikes, selected_expiries):
    row = df[
        df["[EXPIRE_DATE]"].isin(selected_expiries)
        &
        df["[STRIKE]"].isin(selected_strikes)
        ]
    if common_idx:
        common_idx = common_idx.intersection(set(row["[QUOTE_READTIME]"]))
    else:
        common_idx = set(row["[QUOTE_READTIME]"])

days = [item[1:].split(' ')[0] for item in list(common_idx)]



['2023-03-29', '2023-03-28', '2023-03-16', '2023-03-03', '2023-03-31', '2023-03-30', '2023-03-13', '2023-03-21', '2023-03-23', '2023-03-10', '2023-03-17', '2023-03-02', '2023-03-09', '2023-03-27', '2023-03-22', '2023-03-08', '2023-03-06', '2023-03-07', '2023-03-01', '2023-03-24', '2023-03-15', '2023-03-14', '2023-03-20']


In [ ]:
counts = df.groupby(['[QUOTE_UNIXTIME]', '[STRIKE]']).ngroups
counts_strikes = df.groupby(["[STRIKE]"]).ngroups
counts_exps = df.groupby(["[EXPIRE_DATE]"]).ngroups
counts_snapshot_times = df.groupby(["[QUOTE_UNIXTIME]"]).ngroups
print('counts:', counts) # amount of contracts in total (including calls/puts)
print('strike count:', counts_strikes)
print('exp counts:', counts_exps)
print('T:', counts_snapshot_times)
strikes = df.groupby(["[STRIKE]"]).groups.keys()
expiries = df.groupby(["[EXPIRE_DATE]"]).groups.keys()
quotes = df.groupby(["[QUOTE_UNIXTIME]"]).groups.keys()
print('strikes:'); print(strikes)
print('expiries:'); print(expiries)
closest_idx = np.argmin(list(df.groupby(["[STRIKE_DISTANCE]"]).groups.keys()))
print(closest_idx)
closest_strike = list(strikes)[closest_idx]
print(closest_strike)
idx_lowest_quote = np.argmin(list(quotes))
print('starting date:', list(df.groupby(["[QUOTE_READTIME]"]).groups.keys())[idx_lowest_quote])

counts: 6642
strike count: 289
exp counts: 53
T: 23
strikes:
dict_keys([120.0, 130.0, 140.0, 150.0, 155.0, 160.0, 165.0, 170.0, 175.0, 180.0, 185.0, 190.0, 195.0, 200.0, 205.0, 210.0, 215.0, 220.0, 225.0, 230.0, 235.0, 240.0, 245.0, 250.0, 255.0, 260.0, 265.0, 270.0, 275.0, 280.0, 285.0, 290.0, 295.0, 300.0, 301.0, 302.0, 303.0, 304.0, 305.0, 306.0, 307.0, 308.0, 309.0, 310.0, 311.0, 312.0, 313.0, 314.0, 315.0, 316.0, 317.0, 318.0, 319.0, 320.0, 321.0, 322.0, 323.0, 324.0, 325.0, 326.0, 327.0, 328.0, 329.0, 330.0, 331.0, 332.0, 333.0, 334.0, 335.0, 336.0, 337.0, 338.0, 339.0, 340.0, 341.0, 342.0, 343.0, 344.0, 345.0, 346.0, 347.0, 348.0, 349.0, 350.0, 351.0, 352.0, 353.0, 354.0, 355.0, 356.0, 357.0, 358.0, 359.0, 360.0, 361.0, 362.0, 363.0, 364.0, 365.0, 366.0, 367.0, 368.0, 369.0, 370.0, 371.0, 372.0, 373.0, 374.0, 375.0, 376.0, 377.0, 378.0, 379.0, 380.0, 381.0, 382.0, 382.5, 383.0, 384.0, 385.0, 386.0, 387.0, 387.5, 388.0, 389.0, 390.0, 391.0, 392.0, 392.5, 393.0, 394.0, 395.0, 396.

In [31]:
# analysing volume
filtered_df = df[df["[C_VOLUME]"] + df["[P_VOLUME]"] > 1]
volume = filtered_df["[C_VOLUME]"] + filtered_df["[P_VOLUME]"]
print(volume)
print('volume mean:', volume.mean())
print('volume std:', volume.std())
print('volume median:', volume.median())

0         3.0
6         5.0
17       17.0
36        3.0
38        7.0
         ... 
91196     6.0
91200     6.0
91204     9.0
91206    11.0
91209     2.0
Length: 48499, dtype: float64
volume mean: 1453.1622301490752
volume std: 10914.842214044998
volume median: 77.0


In [4]:
# investigate spread of time distance between snapshots
time_dists = []
quotes = df.groupby(["[QUOTE_UNIXTIME]"])
quotes_gns = list(quotes.groups.keys())
differences = np.diff(quotes_gns)
print('avg diff:', np.mean(differences))
print('median diff:', np.median(differences))
# ref point: 100_000 seconds (= 27.78 hours)


avg diff: 117654.54545454546
median diff: 86400.0


In [ ]:
import plotly.graph_objects as go
from datetime import datetime

# investigate spread of expiry dates
min_date = df["[QUOTE_UNIXTIME]"].min()
max_date = df["[QUOTE_UNIXTIME]"].max()
expiries = df["[EXPIRE_DATE]"].to_numpy()




In [5]:
import pandas as pd
import numpy as np

df = df.replace(' ', np.nan).replace('', np.nan)
df = df.fillna(0)
df.columns = df.columns.str.strip()

# Ensure numeric columns are converted to float
numeric_columns = ['[C_VOLUME]', '[P_VOLUME]', '[UNDERLYING_LAST]', '[STRIKE]']
for col in numeric_columns:
    df[col] = df[col].astype(float)

In [6]:
import plotly.graph_objects as go

fig = go.Figure()

groups = df.groupby(["[STRIKE]", "[EXPIRE_DATE]"])
groupnames = list(groups.groups.keys())
for gn in groupnames:
    group = groups.get_group(gn)
    subgr = group.groupby("[QUOTE_UNIXTIME]").agg({
        "[P_VOLUME]": "sum", "[C_VOLUME]": "sum"
        })
    sum_volumes = subgr["[P_VOLUME]"] + subgr["[C_VOLUME]"]
    fig.add_trace(go.Scatter(x=group["[QUOTE_READTIME]"], y=sum_volumes, name=str(gn), mode="markers"))


fig.show()